### ISOT BERT MODEL

In [ ]:
"""
ISOT Fake News Detection with BERT Fine-tuning
=============================================

Complete script for fine-tuning BERT on ISOT dataset with confidence calibration.
Just update the file paths and run!

Dataset: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
Download the dataset and extract Fake.csv and True.csv files.

Author: Your Name
Date: 2025
"""

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# =========================
# CONFIGURATION - UPDATE THESE PATHS
# =========================
# UPDATE THESE PATHS TO YOUR FILES
FAKE_CSV_PATH = "/content/drive/MyDrive/ISOT/Fake.csv"          # Path to Fake.csv
TRUE_CSV_PATH = "/content/drive/MyDrive/ISOT/True.csv"          # Path to True.csv
OUTPUT_DIR = "/content/drive/MyDrive/ISOT/ISOT_Results/"             # Training outputs
SAVE_DIR = "/content/drive/MyDrive/ISOT/ISOT_Model/"                 # Final model

# Model Configuration
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
TEMPERATURE = 1.5  # For confidence calibration
LABEL_SMOOTHING = 0.1  # Reduces overconfidence
SEED = 42

print(f"🎯 ISOT Fake News Detection with BERT")
print(f"📁 Fake news file: {FAKE_CSV_PATH}")
print(f"📁 True news file: {TRUE_CSV_PATH}")
print(f"💾 Model will be saved to: {SAVE_DIR}")

# =========================
# STEP 1: LOAD ISOT DATASET
# =========================
def load_isot_dataset(fake_path, true_path):
    """Load and prepare ISOT fake news dataset"""

    print("🔄 Loading ISOT dataset...")

    try:
        # Load the CSV files
        fake_df = pd.read_csv(fake_path)
        true_df = pd.read_csv(true_path)

        print(f"✅ Loaded fake news: {len(fake_df)} samples")
        print(f"✅ Loaded true news: {len(true_df)} samples")

        # Check columns
        print(f"Fake news columns: {fake_df.columns.tolist()}")
        print(f"True news columns: {true_df.columns.tolist()}")

        # Add labels
        fake_df['label'] = 0  # Fake = 0
        true_df['label'] = 1  # Real = 1

        # Combine title and text for full content
        if 'title' in fake_df.columns and 'text' in fake_df.columns:
            fake_df['full_text'] = fake_df['title'].astype(str) + '. ' + fake_df['text'].astype(str)
            true_df['full_text'] = true_df['title'].astype(str) + '. ' + true_df['text'].astype(str)
            text_column = 'full_text'
        elif 'text' in fake_df.columns:
            text_column = 'text'
        else:
            print("❌ No 'text' or 'title' column found")
            return None

        # Create final dataset
        fake_clean = fake_df[[text_column, 'label']].rename(columns={text_column: 'text'})
        true_clean = true_df[[text_column, 'label']].rename(columns={text_column: 'text'})

        # Combine datasets
        combined_df = pd.concat([fake_clean, true_clean], ignore_index=True)

        # Clean data
        combined_df = combined_df.dropna(subset=['text']).reset_index(drop=True)
        combined_df['text'] = combined_df['text'].astype(str)

        # Remove very short articles (likely noise)
        initial_len = len(combined_df)
        combined_df = combined_df[combined_df['text'].str.len() > 100].reset_index(drop=True)
        removed = initial_len - len(combined_df)

        print(f"🧹 Removed {removed} short articles (< 100 chars)")
        print(f"✅ Final dataset: {len(combined_df)} samples")
        print(f"   📰 Real news: {(combined_df['label'] == 1).sum()}")
        print(f"   🚨 Fake news: {(combined_df['label'] == 0).sum()}")

        return combined_df

    except FileNotFoundError as e:
        print(f"❌ File not found: {e}")
        print("Make sure you've downloaded the ISOT dataset from Kaggle:")
        print("https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset")
        return None
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        return None

# =========================
# STEP 2: CALIBRATED TRAINER (REDUCES OVERCONFIDENCE)
# =========================
class CalibratedBERTTrainer(Trainer):
    """Custom trainer with temperature scaling and label smoothing to reduce overconfidence"""

    def __init__(self, temperature=1.0, label_smoothing=0.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.temperature = temperature
        self.label_smoothing = label_smoothing
        print(f"🌡️  Using temperature scaling: {temperature}")
        print(f"🎯 Using label smoothing: {label_smoothing}")

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """Custom loss with temperature scaling and label smoothing"""
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')

        # Apply temperature scaling
        scaled_logits = logits / self.temperature

        if self.label_smoothing > 0:
            # Label smoothing: instead of [0, 1], use [0.1, 0.9]
            num_classes = logits.size(-1)
            smooth_labels = torch.full_like(scaled_logits, self.label_smoothing / (num_classes - 1))
            smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - self.label_smoothing)

            # KL divergence loss for soft targets
            log_probs = F.log_softmax(scaled_logits, dim=-1)
            loss = F.kl_div(log_probs, smooth_labels, reduction='batchmean')
        else:
            # Standard cross-entropy loss
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(scaled_logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# =========================
# STEP 3: TRAINING FUNCTION
# =========================
def train_isot_fake_news_detector():
    """Train BERT on ISOT dataset with confidence calibration"""

    # Set random seeds for reproducibility
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🚀 Using device: {device}")

    # Load ISOT dataset
    df = load_isot_dataset(FAKE_CSV_PATH, TRUE_CSV_PATH)
    if df is None:
        return None, None

    # Split the data: 70% train, 15% validation, 15% test
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label'])

    print(f"\n📊 Data Splits:")
    print(f"   🏋️  Training: {len(train_df):,} samples (Real: {(train_df['label']==1).sum():,}, Fake: {(train_df['label']==0).sum():,})")
    print(f"   ✅ Validation: {len(val_df):,} samples (Real: {(val_df['label']==1).sum():,}, Fake: {(val_df['label']==0).sum():,})")
    print(f"   🧪 Test: {len(test_df):,} samples (Real: {(test_df['label']==1).sum():,}, Fake: {(test_df['label']==0).sum():,})")

    # Create HuggingFace datasets
    datasets = DatasetDict({
        "train": Dataset.from_pandas(train_df[['text', 'label']], preserve_index=False),
        "validation": Dataset.from_pandas(val_df[['text', 'label']], preserve_index=False),
        "test": Dataset.from_pandas(test_df[['text', 'label']], preserve_index=False)
    })

    # Initialize tokenizer
    print("🔄 Loading tokenizer and tokenizing data...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    def tokenize_function(examples):
        """Tokenize the text data"""
        return tokenizer(
            examples["text"],
            truncation=True,
            padding=False,
            max_length=MAX_LENGTH,
        )

    # Tokenize all datasets
    tokenized_datasets = datasets.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"],
        desc="Tokenizing"
    )

    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format(type="torch")

    # Data collator for dynamic padding
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Load BERT model
    print("🔄 Loading BERT model...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        hidden_dropout_prob=0.3,      # Increased dropout to reduce overconfidence
        attention_probs_dropout_prob=0.3,
        classifier_dropout=0.3,
    )
    model.to(device)

    # Metrics computation function
    def compute_metrics(eval_pred):
        """Compute evaluation metrics"""
        logits, labels = eval_pred

        # Apply temperature scaling for evaluation
        scaled_logits = logits / TEMPERATURE
        probs = torch.softmax(torch.from_numpy(scaled_logits), dim=-1).numpy()
        predictions = np.argmax(probs, axis=-1)

        # Calculate metrics
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average=None, zero_division=0
        )

        accuracy = accuracy_score(labels, predictions)
        macro_f1 = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)[2]

        # Confidence metrics
        max_probs = np.max(probs, axis=1)
        avg_confidence = np.mean(max_probs)

        # Confusion matrix
        cm = confusion_matrix(labels, predictions)

        return {
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "fake_precision": precision[0] if len(precision) > 0 else 0.0,
            "fake_recall": recall[0] if len(recall) > 0 else 0.0,
            "fake_f1": f1[0] if len(f1) > 0 else 0.0,
            "real_precision": precision[1] if len(precision) > 1 else 0.0,
            "real_recall": recall[1] if len(recall) > 1 else 0.0,
            "real_f1": f1[1] if len(f1) > 1 else 0.0,
            "avg_confidence": avg_confidence,
            "tn": int(cm[0, 0]) if cm.shape == (2, 2) else 0,  # True Negatives (correctly predicted fake)
            "fp": int(cm[0, 1]) if cm.shape == (2, 2) else 0,  # False Positives (fake predicted as real)
            "fn": int(cm[1, 0]) if cm.shape == (2, 2) else 0,  # False Negatives (real predicted as fake)
            "tp": int(cm[1, 1]) if cm.shape == (2, 2) else 0,  # True Positives (correctly predicted real)
        }

    # Training arguments
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,  # Larger batch for evaluation
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
        report_to="none",  # Disable wandb logging
        remove_unused_columns=False,
    )

    # Create calibrated trainer
    trainer = CalibratedBERTTrainer(
        temperature=TEMPERATURE,
        label_smoothing=LABEL_SMOOTHING,
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    # Start training
    print("🚀 Starting training...")
    print("=" * 60)
    trainer.train()

    # Evaluate on test set
    print("\n🔄 Evaluating on test set...")
    test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])

    print("\n📊 FINAL TEST RESULTS:")
    print("=" * 50)
    for key, value in test_results.items():
        if isinstance(value, (int, float)):
            if 'confidence' in key:
                print(f"   📈 {key}: {value:.1%}")
            elif key in ['tn', 'fp', 'fn', 'tp']:
                print(f"   🔢 {key}: {value}")
            else:
                print(f"   📊 {key}: {value:.4f}")

    # Save model and tokenizer
    print(f"\n💾 Saving model to {SAVE_DIR}...")
    os.makedirs(SAVE_DIR, exist_ok=True)
    trainer.save_model(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)

    # Save calibration parameters
    calibration_params = {
        'temperature': TEMPERATURE,
        'label_smoothing': LABEL_SMOOTHING,
        'test_results': test_results
    }
    torch.save(calibration_params, os.path.join(SAVE_DIR, 'calibration_params.pt'))

    print("✅ Model and parameters saved successfully!")

    return trainer, tokenizer, test_results

# =========================
# STEP 4: TESTING AND PREDICTION
# =========================
class ISOTFakeNewsPredictor:
    """Predictor class with confidence calibration"""

    def __init__(self, model_dir):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(self.device)

        # Load calibration parameters
        try:
            calib_params = torch.load(os.path.join(model_dir, 'calibration_params.pt'))
            self.temperature = calib_params.get('temperature', 1.5)
            print(f"🌡️  Loaded temperature: {self.temperature}")
        except:
            self.temperature = 1.5
            print(f"🌡️  Using default temperature: {self.temperature}")

    def predict(self, text, return_probabilities=False):
        """Predict fake or real news with calibrated confidence"""

        inputs = self.tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits

            # Apply temperature scaling
            scaled_logits = logits / self.temperature
            probs = torch.softmax(scaled_logits, dim=-1)

            prediction = torch.argmax(probs, dim=-1).item()
            fake_confidence = probs[0, 0].item()
            real_confidence = probs[0, 1].item()

            label = "REAL" if prediction == 1 else "FAKE"
            max_confidence = max(fake_confidence, real_confidence)

            if return_probabilities:
                return {
                    'prediction': label,
                    'confidence': max_confidence,
                    'probabilities': {
                        'fake': fake_confidence,
                        'real': real_confidence
                    }
                }
            else:
                return label, max_confidence

def test_model_samples(model_dir):
    """Test the trained model on sample news articles"""

    predictor = ISOTFakeNewsPredictor(model_dir)

    # Test samples
    test_samples = [
        # Real news samples
        ("The Federal Reserve announced Wednesday it would raise interest rates by 0.25 percentage points, marking the third increase this year as officials continue their fight against inflation. The decision was unanimous among voting members of the Federal Open Market Committee.", "REAL"),

        ("European Union leaders agreed to new sanctions against Russia following the latest developments in the ongoing conflict, according to a statement released after the Brussels summit. The sanctions target key sectors of the Russian economy.", "REAL"),

        ("Scientists at Stanford University have developed a new method for producing hydrogen fuel using solar energy, with results published in the journal Nature showing promising efficiency gains. The breakthrough could revolutionize clean energy production.", "REAL"),

        # Fake news samples
        ("BREAKING: Scientists have discovered that drinking tap water causes immediate brain damage in laboratory studies that were never peer-reviewed by any legitimate scientific institution. Government officials refuse to comment on this shocking revelation.", "FAKE"),

        ("EXCLUSIVE: Pharmaceutical companies are secretly adding mind control chemicals to all vaccines according to leaked documents from insider whistleblowers. Doctors are legally forbidden from discussing this with patients.", "FAKE"),

        ("URGENT: New study proves that smartphones emit deadly radiation that kills brain cells instantly. The government has been covering this up for years to protect tech company profits.", "FAKE"),
    ]

    print("\n🧪 TESTING TRAINED MODEL:")
    print("=" * 70)

    correct_predictions = 0
    total_confidence = 0

    for i, (text, expected_label) in enumerate(test_samples, 1):
        result = predictor.predict(text, return_probabilities=True)
        prediction = result['prediction']
        confidence = result['confidence']
        probs = result['probabilities']

        print(f"\n📰 Sample {i}:")
        print(f"Text: {text[:100]}...")
        print(f"Expected: {expected_label}")
        print(f"Predicted: {prediction} ({confidence:.1%} confidence)")
        print(f"Probabilities: 🚨 Fake: {probs['fake']:.1%}, ✅ Real: {probs['real']:.1%}")

        if prediction == expected_label:
            correct_predictions += 1
            print("✅ CORRECT!")
        else:
            print("❌ INCORRECT")

        total_confidence += confidence

    # Calculate metrics
    accuracy = correct_predictions / len(test_samples)
    avg_confidence = total_confidence / len(test_samples)

    print(f"\n📊 SAMPLE TEST RESULTS:")
    print("=" * 40)
    print(f"🎯 Accuracy: {accuracy:.1%} ({correct_predictions}/{len(test_samples)})")
    print(f"📈 Average Confidence: {avg_confidence:.1%}")

    # Assessment
    if avg_confidence < 85:
        print("✅ EXCELLENT: Well-calibrated confidence!")
    elif avg_confidence < 90:
        print("⚠️  GOOD: Some overconfidence reduction achieved")
    else:
        print("❌ ISSUE: Still overconfident - consider higher temperature")

    if accuracy >= 0.8:
        print("✅ EXCELLENT: High accuracy maintained!")
    else:
        print("⚠️  Consider more training or better data preprocessing")

    return accuracy, avg_confidence

# =========================
# MAIN EXECUTION
# =========================
def main():
    """Main execution function"""

    print("🎯 ISOT FAKE NEWS DETECTION WITH BERT")
    print("=" * 70)
    print("📋 Configuration:")
    print(f"   🤖 Model: {MODEL_NAME}")
    print(f"   📏 Max Length: {MAX_LENGTH}")
    print(f"   🔥 Batch Size: {BATCH_SIZE}")
    print(f"   🔄 Epochs: {EPOCHS}")
    print(f"   📚 Learning Rate: {LEARNING_RATE}")
    print(f"   🌡️  Temperature: {TEMPERATURE}")
    print(f"   🎯 Label Smoothing: {LABEL_SMOOTHING}")
    print("=" * 70)

    # Check if files exist
    if not os.path.exists(FAKE_CSV_PATH):
        print(f"❌ File not found: {FAKE_CSV_PATH}")
        print("Please download ISOT dataset from:")
        print("https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset")
        return

    if not os.path.exists(TRUE_CSV_PATH):
        print(f"❌ File not found: {TRUE_CSV_PATH}")
        print("Please download ISOT dataset from:")
        print("https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset")
        return

    # Step 1: Train the model
    print("\n🚀 STEP 1: Training BERT on ISOT dataset...")
    trainer, tokenizer, test_results = train_isot_fake_news_detector()

    if trainer is None:
        print("❌ Training failed!")
        return

    # Step 2: Test the model
    print("\n🧪 STEP 2: Testing trained model...")
    accuracy, avg_confidence = test_model_samples(SAVE_DIR)

    # Step 3: Summary
    print(f"\n🎉 TRAINING COMPLETE!")
    print("=" * 50)
    print(f"📁 Model saved to: {SAVE_DIR}")
    print(f"🎯 Test Accuracy: {test_results.get('eval_accuracy', 0):.1%}")
    print(f"📈 Average Confidence: {avg_confidence:.1%}")
    print(f"🏆 Macro F1: {test_results.get('eval_macro_f1', 0):.3f}")

    if avg_confidence < 90 and test_results.get('eval_accuracy', 0) > 0.85:
        print("\n✅ SUCCESS: High accuracy with calibrated confidence!")
        print("Your model is ready for production use!")
    else:
        print("\n⚠️  Model trained successfully but may need tuning")
        print("Consider adjusting temperature or training parameters")

if __name__ == "__main__":
    main()

🎯 ISOT Fake News Detection with BERT
📁 Fake news file: /content/drive/MyDrive/ISOT/Fake.csv
📁 True news file: /content/drive/MyDrive/ISOT/True.csv
💾 Model will be saved to: /content/drive/MyDrive/ISOT/ISOT_Model/
🎯 ISOT FAKE NEWS DETECTION WITH BERT
📋 Configuration:
   🤖 Model: bert-base-uncased
   📏 Max Length: 512
   🔥 Batch Size: 8
   🔄 Epochs: 3
   📚 Learning Rate: 2e-05
   🌡️  Temperature: 1.5
   🎯 Label Smoothing: 0.1

🚀 STEP 1: Training BERT on ISOT dataset...
🚀 Using device: cuda
🔄 Loading ISOT dataset...
✅ Loaded fake news: 23481 samples
✅ Loaded true news: 21417 samples
Fake news columns: ['title', 'text', 'subject', 'date']
True news columns: ['title', 'text', 'subject', 'date']
🧹 Removed 443 short articles (< 100 chars)
✅ Final dataset: 44455 samples
   📰 Real news: 21416
   🚨 Fake news: 23039

📊 Data Splits:
   🏋️  Training: 31,118 samples (Real: 14,991, Fake: 16,127)
   ✅ Validation: 6,668 samples (Real: 3,212, Fake: 3,456)
   🧪 Test: 6,669 samples (Real: 3,213, Fake: 3,4

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🌡️  Using temperature scaling: 1.5
🎯 Using label smoothing: 0.1
🚀 Starting training...


Step,Training Loss,Validation Loss,Accuracy,Macro F1,Fake Precision,Fake Recall,Fake F1,Real Precision,Real Recall,Real F1,Avg Confidence,Tn,Fp,Fn,Tp
500,0.063200,0.226493,0.827385,0.824346,0.993576,0.671296,0.801243,0.737826,0.995330,0.847449,0.856615,2320,1136,15,3197
1000,0.005700,0.026242,0.984553,0.984546,1.000000,0.970197,0.984873,0.968929,1.000000,0.984219,0.910821,3353,103,0,3212
1500,0.003000,0.021497,0.988602,0.988595,1.000000,0.978009,0.988882,0.976886,1.000000,0.988308,0.914377,3380,76,0,3212
2000,0.000900,0.010830,0.996101,0.996097,1.000000,0.992477,0.996224,0.991970,1.000000,0.995969,0.920974,3430,26,0,3212
2500,0.003000,0.005006,0.999400,0.999399,1.000000,0.998843,0.999421,0.998756,1.000000,0.999378,0.924262,3452,4,0,3212
3000,0.000700,0.010965,0.996251,0.996247,1.000000,0.992766,0.996370,0.992277,1.000000,0.996123,0.921770,3431,25,0,3212
3500,0.003000,0.006074,0.999250,0.999249,1.000000,0.998553,0.999276,0.998446,1.000000,0.999222,0.926211,3451,5,0,3212



🔄 Evaluating on test set...



📊 FINAL TEST RESULTS:
   📊 eval_loss: 0.0059
   📊 eval_accuracy: 0.9988
   📊 eval_macro_f1: 0.9988
   📊 eval_fake_precision: 0.9988
   📊 eval_fake_recall: 0.9988
   📊 eval_fake_f1: 0.9988
   📊 eval_real_precision: 0.9988
   📊 eval_real_recall: 0.9988
   📊 eval_real_f1: 0.9988
   📈 eval_avg_confidence: 92.4%
   📊 eval_tn: 3452.0000
   📊 eval_fp: 4.0000
   📊 eval_fn: 4.0000
   📊 eval_tp: 3209.0000
   📊 eval_runtime: 46.6914
   📊 eval_samples_per_second: 142.8310
   📊 eval_steps_per_second: 8.9310
   📊 epoch: 0.8997

💾 Saving model to /content/drive/MyDrive/ISOT/ISOT_Model/...
✅ Model and parameters saved successfully!

🧪 STEP 2: Testing trained model...
🌡️  Loaded temperature: 1.5

🧪 TESTING TRAINED MODEL:

📰 Sample 1:
Text: The Federal Reserve announced Wednesday it would raise interest rates by 0.25 percentage points, mar...
Expected: REAL
Predicted: REAL (88.6% confidence)
Probabilities: 🚨 Fake: 11.4%, ✅ Real: 88.6%
✅ CORRECT!

📰 Sample 2:
Text: European Union leaders agreed to new s

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
pip install huggingface_hub

In [10]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add t

In [11]:
from huggingface_hub import HfApi, login

# You're already logged in, but just to be sure
login()

# Create the repository
api = HfApi()
try:
    repo_url = api.create_repo(
        repo_id="naheelkk/fake-news-bert-isot",
        repo_type="model",
        private=False,  # Set to True if you want it private
        exist_ok=True   # Won't error if repo already exists
    )
    print(f"Repository created/exists: {repo_url}")
except Exception as e:
    print(f"Error creating repo: {e}")

Repository created/exists: https://huggingface.co/naheelkk/fake-news-bert-isot


In [12]:
from huggingface_hub import HfApi
import os

# First download your model files from Drive to local
# Then upload to HF Hub
api = HfApi()
api.upload_folder(
    folder_path="/content/drive/MyDrive/ISOT/ISOT_Model/",
    repo_id="naheelkk/fake-news-bert-isot",
    repo_type="model"
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...e/ISOT/ISOT_Model/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...OT/ISOT_Model/calibration_params.pt:  11%|#         |   206B / 1.90kB            

  ...e/ISOT/ISOT_Model/training_args.bin:  11%|#         |   626B / 5.78kB            

CommitInfo(commit_url='https://huggingface.co/naheelkk/fake-news-bert-isot/commit/f5aa495e46a8434a17f10be3b29fdd396fa3a950', commit_message='Upload folder using huggingface_hub', commit_description='', oid='f5aa495e46a8434a17f10be3b29fdd396fa3a950', pr_url=None, repo_url=RepoUrl('https://huggingface.co/naheelkk/fake-news-bert-isot', endpoint='https://huggingface.co', repo_type='model', repo_id='naheelkk/fake-news-bert-isot'), pr_revision=None, pr_num=None)